In [2]:
import ee

print("Earth Engine:", ee.__version__)

Earth Engine: 1.7.43


In [3]:
import ee

ee.Authenticate()

Enter verification code:  4/1ATsMZqAd07QrTUAdQm_bU2oLsqBz9Ndv4OKmbcWZazrVvn2-dAZEDwXbLF0



Successfully saved authorization token.


In [4]:
ee.Initialize()
print("Earth Engine conectado!")

Earth Engine conectado!


In [5]:
'
# 05 — Teste de conexão com o Google Earth Engine
# ============================================================

import ee

ee.Initialize()

print("✓ Google Earth Engine conectado com sucesso!")

✓ Google Earth Engine conectado com sucesso!


In [6]:
# ============================================================
# 06 — Primeiro acesso ao catálogo Sentinel-2
# ============================================================

colecao = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterDate("2024-01-01", "2024-12-31")
)

print("Número de imagens:", colecao.size().getInfo())

Número de imagens: 4295362


In [2]:
# ============================================================
# 07 — Verificar a estrutura atual de data/raw
# ============================================================

pasta_raw = PROJETO / "data" / "raw"

print("Conteúdo de data/raw:\n")

for item in sorted(pasta_raw.iterdir()):
    tipo = "📁" if item.is_dir() else "📄"
    print(f"{tipo} {item.name}")

NameError: name 'PROJETO' is not defined

In [4]:
from pathlib import Path

# A pasta do notebook está dentro de "notebooks"
PROJETO = Path.cwd().parent

print("Pasta do projeto:")
print(PROJETO)

Pasta do projeto:
C:\Users\bbert_xwhb76k\analises-GIS\01-monitoramento-vegetacao


In [5]:
# Verificar o conteúdo de data/raw

pasta_raw = PROJETO / "data" / "raw"

print("Conteúdo de data/raw:\n")

for item in sorted(pasta_raw.iterdir()):
    tipo = "📁" if item.is_dir() else "📄"
    print(f"{tipo} {item.name}")

Conteúdo de data/raw:

📄 Area_de_Preservacao_Permanente.zip
📄 Area_de_Uso_Restrito.zip
📄 Area_do_Imovel.zip
📄 Cobertura_do_Solo.zip
📄 MARCADORES_Area_de_Preservacao_Permanente.zip
📄 Reserva_Legal.zip


In [6]:
# ============================================================
# 08 — Extração das camadas do CAR
# ============================================================

import zipfile

arquivos_car = sorted(pasta_raw.glob("*.zip"))

for arquivo_zip in arquivos_car:
    
    # Nome da pasta = nome do ZIP sem ".zip"
    pasta_saida = pasta_raw / arquivo_zip.stem
    
    # Cria a pasta caso ainda não exista
    pasta_saida.mkdir(exist_ok=True)
    
    # Extrai o conteúdo
    with zipfile.ZipFile(arquivo_zip, "r") as z:
        z.extractall(pasta_saida)
    
    print(f"✓ {arquivo_zip.name} → {pasta_saida.name}/")

✓ Area_de_Preservacao_Permanente.zip → Area_de_Preservacao_Permanente/
✓ Area_de_Uso_Restrito.zip → Area_de_Uso_Restrito/
✓ Area_do_Imovel.zip → Area_do_Imovel/
✓ Cobertura_do_Solo.zip → Cobertura_do_Solo/
✓ MARCADORES_Area_de_Preservacao_Permanente.zip → MARCADORES_Area_de_Preservacao_Permanente/
✓ Reserva_Legal.zip → Reserva_Legal/


In [7]:
# ============================================================
# 09 — Localização dos arquivos geográficos
# ============================================================

shapefiles = sorted(pasta_raw.rglob("*.shp"))

print(f"Total de Shapefiles encontrados: {len(shapefiles)}\n")

for shp in shapefiles:
    print("✓", shp.relative_to(pasta_raw))

Total de Shapefiles encontrados: 6

✓ Area_de_Preservacao_Permanente\Area_de_Preservacao_Permanente.shp
✓ Area_de_Uso_Restrito\Area_de_Uso_Restrito.shp
✓ Area_do_Imovel\Area_do_Imovel.shp
✓ Cobertura_do_Solo\Cobertura_do_Solo.shp
✓ MARCADORES_Area_de_Preservacao_Permanente\MARCADORES_Area_de_Preservacao_Permanente.shp
✓ Reserva_Legal\Reserva_Legal.shp


In [8]:
# ============================================================
# 10 — Leitura das camadas do CAR com GeoPandas
# ============================================================

import geopandas as gpd

camadas = {}

for shp in shapefiles:
    nome = shp.stem
    
    gdf = gpd.read_file(shp)
    
    camadas[nome] = gdf
    
    print(f"\n{'=' * 60}")
    print(nome)
    print(f"{'=' * 60}")
    print(f"Feições: {len(gdf)}")
    print(f"CRS: {gdf.crs}")
    print(f"Geometrias: {gdf.geometry.geom_type.unique()}")


Area_de_Preservacao_Permanente
Feições: 10
CRS: EPSG:4674
Geometrias: <StringArray>
['MultiPolygon', 'Polygon']
Length: 2, dtype: str

Area_de_Uso_Restrito
Feições: 1
CRS: EPSG:4674
Geometrias: <StringArray>
['MultiPolygon']
Length: 1, dtype: str

Area_do_Imovel
Feições: 2
CRS: EPSG:4674
Geometrias: <StringArray>
['Polygon']
Length: 1, dtype: str

Cobertura_do_Solo
Feições: 2
CRS: EPSG:4674
Geometrias: <StringArray>
['MultiPolygon']
Length: 1, dtype: str

MARCADORES_Area_de_Preservacao_Permanente
Feições: 1
CRS: EPSG:4674
Geometrias: <StringArray>
['MultiPoint']
Length: 1, dtype: str

Reserva_Legal
Feições: 2
CRS: EPSG:4674
Geometrias: <StringArray>
['MultiPolygon']
Length: 1, dtype: str


In [9]:
# ============================================================
# 11 — Inspeção dos atributos das camadas
# ============================================================

for nome, gdf in camadas.items():
    
    print("\n" + "=" * 70)
    print(nome)
    print("=" * 70)
    
    print("\nColunas:")
    print(list(gdf.columns))
    
    print("\nPrimeiros registros:")
    display(gdf.head())


Area_de_Preservacao_Permanente

Colunas:
['recibo', 'area', 'tema', 'geometry']

Primeiros registros:


,recibo,area,tema,geometry
0,SP-3552205-CBC00ACCA5874AC3B43BF5DF37F26251,4.7082,Ãrea de PreservaÃ§Ã£o Permanente em Ã¡rea con...,"MULTIPOLYGON (((-47.53517 -23.46637, -47.53517..."
1,SP-3552205-CBC00ACCA5874AC3B43BF5DF37F26251,9.3823,Ãrea de PreservaÃ§Ã£o Permanente em Ã¡rea de ...,"MULTIPOLYGON (((-47.54099 -23.46721, -47.54096..."
2,SP-3552205-CBC00ACCA5874AC3B43BF5DF37F26251,4.1952,APP segundo art. 61-A da Lei nÂº 12.651/2012,"MULTIPOLYGON (((-47.53546 -23.46743, -47.53547..."
3,SP-3552205-CBC00ACCA5874AC3B43BF5DF37F26251,0.0111,Ãrea de PreservaÃ§Ã£o Permanente a Recompor d...,"POLYGON ((-47.53546 -23.46743, -47.53547 -23.4..."
4,SP-3552205-CBC00ACCA5874AC3B43BF5DF37F26251,4.2318,Ãrea de PreservaÃ§Ã£o Permanente a Recompor d...,"MULTIPOLYGON (((-47.53455 -23.46633, -47.53458..."



Area_de_Uso_Restrito

Colunas:
['recibo', 'area', 'tema', 'geometry']

Primeiros registros:


,recibo,area,tema,geometry
0,SP-3552205-CBC00ACCA5874AC3B43BF5DF37F26251,183.1718,Ãrea de Uso Restrito para declividade de 25 a...,"MULTIPOLYGON (((-47.53421 -23.48161, -47.53431..."



Area_do_Imovel

Colunas:
['recibo', 'modfiscais', 'tema', 'area', 'municipio', 'estado', 'geometry']

Primeiros registros:


...

,recibo,area,tema,geometry
0,SP-3552205-CBC00ACCA5874AC3B43BF5DF37F26251,169.7244,Ãrea Consolidada,"MULTIPOLYGON (((-47.53517 -23.46637, -47.53517..."
1,SP-3552205-CBC00ACCA5874AC3B43BF5DF37F26251,18.9554,Remanescente de VegetaÃ§Ã£o Nativa,"MULTIPOLYGON (((-47.53488 -23.46619, -47.53487..."



MARCADORES_Area_de_Preservacao_Permanente

Colunas:
['recibo', 'area', 'tema', 'geometry']

Primeiros registros:


,recibo,area,tema,geometry
0,SP-3552205-CBC00ACCA5874AC3B43BF5DF37F26251,0.0,Nascente ou olho d'Ã¡gua perene,MULTIPOINT (-47.5356 -23.46743)



Reserva_Legal

Colunas:
['recibo', 'area', 'tema', 'geometry']

Primeiros registros:


,recibo,area,tema,geometry
0,SP-3552205-CBC00ACCA5874AC3B43BF5DF37F26251,18.9554,Reserva Legal Proposta,"MULTIPOLYGON (((-47.53488 -23.46619, -47.53487..."
1,SP-3552205-CBC00ACCA5874AC3B43BF5DF37F26251,18.9554,Ãrea de Reserva Legal Total,"MULTIPOLYGON (((-47.53488 -23.46619, -47.53487..."


In [11]:
# ============================================================
# 12. Resumo das camadas do CAR
# ============================================================

import pandas as pd

resumo = []

for nome, gdf in camadas.items():
    resumo.append({
        "camada": nome,
        "feicoes": len(gdf),
        "crs": str(gdf.crs),
        "geometrias": ", ".join(gdf.geometry.geom_type.unique()),
        "area_atributo_ha": gdf["area"].sum() if "area" in gdf.columns else None
    })

resumo_car = pd.DataFrame(resumo)

display(resumo_car)

,camada,feicoes,crs,geometrias,area_atributo_ha
0,Area_de_Preservacao_Permanente,10,EPSG:4674,"MultiPolygon, Polygon",51.1762
1,Area_de_Uso_Restrito,1,EPSG:4674,MultiPolygon,183.1718
2,Area_do_Imovel,2,EPSG:4674,Polygon,377.8392
3,Cobertura_do_Solo,2,EPSG:4674,MultiPolygon,188.6798
4,MARCADORES_Area_de_Preservacao_Permanente,1,EPSG:4674,MultiPoint,0.0000
5,Reserva_Legal,2,EPSG:4674,MultiPolygon,37.9108


In [12]:
# ============================================================
# 13. Verificação das geometrias por registro temático
# ============================================================

for nome in ["Area_do_Imovel", "Reserva_Legal"]:
    gdf = camadas[nome]

    print(f"\n{'=' * 60}")
    print(nome)
    print(f"{'=' * 60}")

    for i, row in gdf.iterrows():
        print(f"\nRegistro {i}:")
        print("Tema:", row["tema"])
        print("Área declarada:", row["area"], "ha")
        print("Tipo:", row.geometry.geom_type)
        print("Válida:", row.geometry.is_valid)

    if len(gdf) == 2:
        mesma_geometria = gdf.geometry.iloc[0].equals(gdf.geometry.iloc[1])
        print("\nAs duas geometrias são iguais?", mesma_geometria)


Area_do_Imovel

Registro 0:
Tema: Ãrea do Imovel
Área declarada: 188.9196 ha
Tipo: Polygon
Válida: True

Registro 1:
Tema: Ãrea LÃ­quida do ImÃ³vel
Área declarada: 188.9196 ha
Tipo: Polygon
Válida: True

As duas geometrias são iguais? True

Reserva_Legal

Registro 0:
Tema: Reserva Legal Proposta
Área declarada: 18.9554 ha
Tipo: MultiPolygon
Válida: True

Registro 1:
Tema: Ãrea de Reserva Legal Total
Área declarada: 18.9554 ha
Tipo: MultiPolygon
Válida: True

As duas geometrias são iguais? True


In [13]:
# ============================================================
# 14. Reprojeção para cálculo de área
# ============================================================

camadas_proj = {
    nome: gdf.to_crs("EPSG:31983")
    for nome, gdf in camadas.items()
}

print("✓ Camadas reprojetadas para EPSG:31983")
print("✓ SIRGAS 2000 / UTM 23S")

✓ Camadas reprojetadas para EPSG:31983
✓ SIRGAS 2000 / UTM 23S


In [14]:
# ============================================================
# 15. Comparação entre área declarada e área da geometria
# ============================================================

comparacao = []

for nome, gdf in camadas_proj.items():
    for i, row in gdf.iterrows():
        
        area_geom_ha = row.geometry.area / 10_000
        
        comparacao.append({
            "camada": nome,
            "registro": i,
            "tema": row["tema"] if "tema" in row else None,
            "area_declarada_ha": row["area"] if "area" in row else None,
            "area_geometria_ha": area_geom_ha,
            "diferenca_ha": (
                area_geom_ha - row["area"]
                if "area" in row else None
            )
        })

comparacao_areas = pd.DataFrame(comparacao)

display(comparacao_areas)

,camada,registro,tema,area_declarada_ha,area_geometria_ha,diferenca_ha
0,Area_de_Preservacao_Permanente,0,Ãrea de PreservaÃ§Ã£o Permanente em Ã¡rea con...,4.7082,4.708193,-0.000007
1,Area_de_Preservacao_Permanente,1,Ãrea de PreservaÃ§Ã£o Permanente em Ã¡rea de ...,9.3823,9.382347,0.000047
2,Area_de_Preservacao_Permanente,2,APP segundo art. 61-A da Lei nÂº 12.651/2012,4.1952,4.195249,0.000049
3,Area_de_Preservacao_Permanente,3,Ãrea de PreservaÃ§Ã£o Permanente a Recompor d...,0.0111,0.011108,0.000008
4,Area_de_Preservacao_Permanente,4,Ãrea de PreservaÃ§Ã£o Permanente a Recompor d...,4.2318,4.184141,-0.047659
5,Area_de_Preservacao_Permanente,5,Ãrea de PreservaÃ§Ã£o Permanente de Nascentes...,0.7131,0.713146,0.000046
6,Area_de_Preservacao_Permanente,6,Ãrea de PreservaÃ§Ã£o Permanente de Rios atÃ©...,13.6042,13.394045,-0.210155
7,Area_de_Preservacao_Permanente,7,APP Total,14.0905,14.090547,0.000047
8,Area_de_Preservacao_Permanente,8,Ãrea de PreservaÃ§Ã£o Permanente em Ã¡rea ant...,0.0000,0.000008,0.000008
9,Area_de_Preservacao_Permanente,9,Curso d'Ã¡gua natural de atÃ© 10 metros,0.2398,0.239763,-0.000037


In [15]:
# ============================================================
# 16. Temas e áreas das APPs
# ============================================================

app = camadas["Area_de_Preservacao_Permanente"]

app_temas = app[["tema", "area"]].copy()

app_temas.columns = ["tema", "area_ha"]

display(app_temas)

,tema,area_ha
0,Ãrea de PreservaÃ§Ã£o Permanente em Ã¡rea con...,4.7082
1,Ãrea de PreservaÃ§Ã£o Permanente em Ã¡rea de ...,9.3823
2,APP segundo art. 61-A da Lei nÂº 12.651/2012,4.1952
3,Ãrea de PreservaÃ§Ã£o Permanente a Recompor d...,0.0111
4,Ãrea de PreservaÃ§Ã£o Permanente a Recompor d...,4.2318
5,Ãrea de PreservaÃ§Ã£o Permanente de Nascentes...,0.7131
6,Ãrea de PreservaÃ§Ã£o Permanente de Rios atÃ©...,13.6042
7,APP Total,14.0905
8,Ãrea de PreservaÃ§Ã£o Permanente em Ã¡rea ant...,0.0000
9,Curso d'Ã¡gua natural de atÃ© 10 metros,0.2398
